1.1 Sparse attention

In [2]:
import torch
from torch import nn
import torch.nn.functional as F
class SparseAttention(nn.Module):
    def __init__(self,d_in,d_out):
        super().__init__()

        self.W_query=nn.Linear(d_in,d_out,bias=False)
        self.W_key=nn.Linear(d_in,d_out,bias=False)
        self.W_value=nn.Linear(d_in,d_out,bias=False)

        def forward(self,x):
            queries=self.W_query(x)
            keys=self.W_key(x)
            values=self.W_value(x)
            attn_scores=torch.matmul(queries,keys.transpose(-2,-1))
            attn_scores=attn_scores/keys.shape[-1]**0.5
            seq_len=attn_scores.shape[0]
            mask=torch.tril(torch.ones(seq_len,seq_len))
            attn_scores=attn_scores.masked_fill(mask==0,float("-inf"))
            attn_weights=torch.softmax(attn_scores,dim=-1)
            context_vect=torch.matmul(attn_weights,values)
            return context_vect


1.2 Using sliding window and block-sparse

In [6]:
class SlidingWindow(nn.Module):
    def __init__(self,d_in,d_out):
        super().__init__()

        self.W_query=nn.Linear(d_in,d_out,bias=False)
        self.W_key=nn.Linear(d_in,d_out,bias=False)
        self.W_value=nn.Linear(d_in,d_out,bias=False)

        def forward(self,x):
            queries=self.W_query(x)
            keys=self.W_key(x)
            values=self.W_value(x)
            window=2
            attn_scores=torch.matmul(queries,keys.transpose(-2,-1))
            attn_scores=attn_scores/keys.shape[-1]**0.5
            seq_len=attn_scores.shape[-1]
            mask=torch.zeros(seq_len,seq_len)
            for i in range(seq_len):
                start=max(0,i-window)
                mask[i,start:i+1]=1
            print(mask)
            attn_scores=attn_scores.masked_fill(mask==0,float("-inf"))
            attn_weights=torch.softmax(attn_scores,dim=-1)
            context_vect=torch.matmul(attn_weights,values)
            return context_vect

In [7]:
import random
class BigBirdAttention(nn.Module):

    def __init__(self,d_in,d_out):
        super().__init__()

        self.W_query=nn.Linear(d_in,d_out,bias=False)
        self.W_key=nn.Linear(d_in,d_out,bias=False)
        self.W_value=nn.Linear(d_in,d_out,bias=False)

        def forward(self,x):
            queries=self.W_query(x)
            keys=self.W_key(x)
            values=self.W_value(x)
            window=2
            attn_scores=torch.matmul(queries,keys.transpose(-2,-1))
            attn_scores=attn_scores/keys.shape[-1]**0.5
            seq_len=attn_scores.shape[-1]
            #Local Attention
            mask=torch.zeros(seq_len,seq_len)
            for i in range(seq_len):
                start=max(0,i-window)
                mask[i,start:i+1]=1
            print(mask)
            #Global Attention
            mask[:, 0]=1
            mask[0, :]=1
            #Random Attention
            for i in range(seq_len):
               rand_inx=random.sample(range(seq_len),2)
               for j in rand_inx:
                   mask[i,j]=1
            print(mask)
            attn_scores=attn_scores.masked_fill(mask==0,float("-inf"))
            attn_weights=torch.softmax(attn_scores,dim=-1)
            context_vect=torch.matmul(attn_weights,values)
            return context_vect


1.3 Correctness Harness


In [ ]:
import torch
seq_len=10
d_out=63
queries=torch.randn(seq_len,d_out)
keys=torch.randn(seq_len,d_out)
dense_scores=torch.matmul(queries,keys.transpose(-2,-1))
dense_scores=dense_scores/keys.shape[-1]**0.5
dense_mask=torch.tril(torch.zeros(seq_len,seq_len))
dense_scores=dense_scores.masked_fill(dense_mask==0,float("-inf"))
window=2
sparse_mask=torch.zeros(seq_len,seq_len)
for i in range(seq_len):
  start=max(0,i-window)
  for j in range(seq_len):
    sparse_max[i,start:i+1]=1
sparse_scores=torch.matmul(queries,keys.transpose(-2,-1))
sparse_scores=sparse_scores/keys.shape[-1]**0.5
sparse_scores=dense_scores.masked_fill(sparse_mask==0,float("-inf"))
dense_specific=dense_scores[sparse_mask==1]
sparse_specific=sparse_scores[sparse_mask==1]
corre=torch.allclose(dense_specific,sparse_specific,alot=1e-4)
print("PASS=",passed)
if not passed:
  max_error=torch.max(torch.abs(dense_specific,sparse_specific))
  print("Max Error=", max_error)





1.4 Nan Handling

In [ ]:
import torch
seq_len=4
scores=torch.tensor([[-float("inf"),-float("-inf"),-float("-inf"),-float("-inf")]])
